## Test text-to-speech and audio output

In [1]:
import numpy as np
from scipy.signal import resample
from kokoro import KPipeline
from reachy_mini import ReachyMini

KOKORO_SAMPLE_RATE = 24000  # fixed, native output rate of Kokoro

C:\Users\User\Documents\Coding\chess-assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mini = ReachyMini()

In [9]:
from chess_assistant.robot import Speaker

In [10]:
speaker = Speaker(mini)

In [13]:
from dotenv import load_dotenv
import anthropic

In [14]:
client = anthropic.Anthropic()

In [15]:
move = "e3e6"
piece = "Queen"
rating = 100

In [21]:
PROMPT_START = (
    "You are a witty, British chess commentator. "
    "I will provide you with a move, the centipawn loss associated with that move (higher means move was worse; best is zero), "
    "and the piece that moved."
    "Based on that, return a short comment. Applaud solid moves, and roast bad ones."
    "Don't quote the actual centipawn loss number. The listeners won't know what they mean."
    "Return only the comment."
)
prompt = f"{PROMPT_START}\nMove: {move}\nMoved piece: {piece}\nCentipawn loss: {rating}"
message = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=128,
    messages=[{
        "role": "user",
        "content": prompt
    }]
)

'Ah, a hundred-point blunder disguised as a queen move. One might say Her Majesty has abdicated her usual standards of propriety. Frightfully sloppy.'

AttributeError: 'tuple' object has no attribute 'strip'

In [3]:
def format_uci_for_speech(uci_move: str) -> str:
    """Turn 'e2e4' into 'E2 to E4' so it's said clearly instead of
    being mangled as one run-together token."""
    origin, destination = uci_move[:2], uci_move[2:4]
    return f"{origin.upper()} to {destination.upper()}"


In [4]:
def synthesize(pipeline, text: str, voice: str = "af_heart", speed: float = 1.0) -> np.ndarray:
    """Run text through Kokoro and return one concatenated float32 waveform."""
    chunks = []
    for _, _, audio in pipeline(text, voice=voice, speed=speed):
        chunks.append(audio.numpy() if hasattr(audio, "numpy") else audio)
    return np.concatenate(chunks)

In [17]:
def say(mini: ReachyMini, pipeline, text: str, voice: str = "bm_george"):
    """Synthesize text with Kokoro and play it through Reachy Mini's speaker."""
    audio = synthesize(pipeline, text, voice=voice)

    target_rate = mini.media.get_output_audio_samplerate()
    if target_rate != KOKORO_SAMPLE_RATE:
        audio = resample(audio, int(len(audio) * target_rate / KOKORO_SAMPLE_RATE))

    mini.media.start_playing()
    mini.media.push_audio_sample(audio.astype(np.float32))
    import time
    time.sleep(len(audio) / target_rate)
    mini.media.stop_playing()


In [6]:
pipeline = KPipeline(lang_code="b")  # 'b' = British English

C:\Users\User\Documents\Coding\chess-assistant\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--hexgrad--Kokoro-82M. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\User\Documents\Coding\chess-assistant\.venv\Lib\site-packages\torch\nn\modules\rnn.py:100

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [20]:
import time
start = time.perf_counter()
time.sleep(1)
end = time.perf_counter()

In [21]:
end - start

1.000748399994336

In [18]:
uci_move = "e2e4"
text = f"{format_uci_for_speech(uci_move)}?"
say(mini, pipeline, text)

In [ ]:
mini.__exit__()

## Test using antennas as input

In [1]:
from reachy_mini import ReachyMini
import numpy as np

In [2]:
mini = ReachyMini()

In [3]:
_, antenna_positions = mini.get_current_joint_positions()
np.rad2deg(antenna_positions)

array([-120.49804688,  100.28320313])

In [4]:
mini.goto_target(antennas=np.deg2rad([-120, 100]), duration=0.5) 
    # right left; from robots perspective
    # left right, when looking onto robot
    # And then angles are clockwise from our perspective

In [5]:
from chess_assistant.antennas_input import AntennasInputDetector

In [6]:
from pathlib import Path
calibration_metadata_path = Path("data/generated/2026-07-07_082513/calibration_metadata.json")
    # In this case we have top-left in ["a1", "h8"] as desired
    # tl = h8, so side_to_play = right
input_detector = AntennasInputDetector(mini, calibration_metadata_path)

In [7]:
input_detector.side_to_play

'right'

In [8]:
import time

In [9]:
time.sleep()

TypeError: time.sleep() takes exactly one argument (0 given)

In [10]:
input_detector.side_to_play = "left"
move_made = False
while not move_made:
    move_made = input_detector.detect_input("move_made")
    print(f"Move made: {move_made}")
    time.sleep(1)

85.693359375
TEST
Move made: False
85.693359375
TEST
Move made: False
85.693359375
TEST
Move made: False
85.166015625
TEST
Move made: False
85.166015625
TEST
Move made: False
96.94335937500001
TEST
Move made: True


In [13]:
input_detector.side_to_play = "right"
estimated_move_rejected = False
while not estimated_move_rejected:
    estimated_move_rejected = input_detector.detect_input("estimated_move_rejected")
    print(f"Move made: {estimated_move_rejected}")
    time.sleep(1)

-85.517578125
TEST
Move made: False
-85.517578125
TEST
Move made: False
-57.480468749999986
TEST
Move made: True


In [ ]:
mini.__exit__(None, None, None)